# 낙찰가 모델 (코랩)
기준선 → 선형 분위수 회귀 → LightGBM 분위수 회귀 (P10/P50/P90)

**실행 전 준비 (한 번만)**
1. 구글 드라이브 `내 드라이브/신보프로젝트/data/`에 `train_features_20261105.csv`가 있는지 확인해요.
2. 지가 파일 2개(`(월) 지역별 지가변동률.csv`, `(월) 지역별 지가지수.csv`)를 드라이브 `내 드라이브/신보프로젝트/지역별 지가지수, 지가변동률/` 폴더에 올려요. `data` 폴더 안에 넣어도 돼요. 없으면 지가변동률 피처 없이 돌아가요.

**실행**: 위에서부터 순서대로 ▶ (또는 런타임 → 모두 실행). 런타임이 끊기면 처음부터 다시 실행하면 돼요.

## 준비. 드라이브 연결 + 패키지

In [24]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q lightgbm statsmodels
import lightgbm, statsmodels
print("lightgbm", lightgbm.__version__, "/ statsmodels", statsmodels.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
lightgbm 4.6.0 / statsmodels 0.15.0


## 0단계. 설정
`USE_PRPT`(재산유형 피처), `CQR_BY_USAGE`(용도별 구간 보정)를 여기서 바꿔요.

In [25]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ▼ 구글 드라이브 경로 (다르면 여기만 고치기)
PROJECT_DIR = "/content/drive/MyDrive/신보프로젝트"
DATA_DIR = os.path.join(PROJECT_DIR, "data")                         # train_features_*.csv 가 있는 곳
LAND_DIR = next((d for d in [os.path.join(PROJECT_DIR, "지역별 지가지수, 지가변동률"),
                             os.path.join(DATA_DIR, "지역별 지가지수, 지가변동률"),
                             DATA_DIR]
                 if os.path.exists(os.path.join(d, "(월) 지역별 지가변동률.csv"))), None)
OUT_DIR = os.path.join(DATA_DIR, "model_out")
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_END, VAL_END, TEST_END = "2026-08-31", "2026-09-30", "2026-10-31"
QS = [0.1, 0.5, 0.9]
SEED = 42

# ★ 팀 결정 사항: 재산유형(압류/기타일반)을 피처로 쓸지. 5단계에서 둘 다 val 성능을 보여줌
USE_PRPT = True
# 구간 보정(CQR)을 용도별로 따로 할지 (건수 30건 미만 용도는 전체 보정폭 사용)
CQR_BY_USAGE = True
SINBO_USAGE = ["공장·창고", "상가·업무", "대지", "농지·임야"]     # 신보 담보에 가까운 용도 → 대표 성적표

print("data 폴더:", DATA_DIR)
print("지가 폴더:", LAND_DIR or "없음 ⚠️ (지가변동률 피처 없이 진행)")

data 폴더: /content/drive/MyDrive/신보프로젝트/data
지가 폴더: /content/drive/MyDrive/신보프로젝트/data


## 1단계. 데이터 정리
누수 열 제거, 중복·극단값 정리, 용도 6묶음

In [26]:
path = sorted(glob.glob(os.path.join(DATA_DIR, "train_features_*.csv")))[-1]
raw = pd.read_csv(path, low_memory=False, dtype={"sgg_cd": str, "pnu": str})
print("입력:", os.path.basename(path), raw.shape)

# 결과 정보(누수) — 피처로 절대 쓰지 않음
LEAK = ["scfbAmt", "y_ratio_api", "pbctNsq", "lowstBidPrcIndctCont", "apslPrcCtrsLowstBidRto",
        "vldBddrNope", "nfctBddrNope"]

df = raw.copy()
df["opbd_date"] = pd.to_datetime(df["opbd_date"], errors="coerce")
df = df.dropna(subset=["opbd_date", "y_ratio", "apslEvlAmt"])
df = df.sort_values("opbd_date").drop_duplicates("cltrMngNo", keep="last")      # 같은 물건은 최근 개찰 1건

# 극단값: 처음엔 잘라내기(0.05~1.5). 뺀 건수는 기록
n0 = len(df)
df = df[(df["y_ratio"] >= 0.05) & (df["y_ratio"] <= 1.5)].copy()
print(f"중복·결측·극단값 정리: {len(raw)} → {n0} → {len(df)}건")

# 용도 6묶음 (11/01 정리본 기준, 경매 매각통계·경매 사례와 짝을 맞추는 단위)
def usage6(m, s):
    s = str(s)
    if s == "아파트":
        return "아파트"
    if m == "주거용건물":
        return "기타주거"
    if s in ("공장시설", "기타제조공장", "창고시설", "공장용지", "창고용지"):
        return "공장·창고"
    if m == "토지":
        if s in ("대지", "잡종지", "주차장", "주유소용지", "종교용지", "수도용지", "기타토지", "도로", "공원", "구거", "하천", "묘지", "염전"):
            return "대지"
        return "농지·임야"                    # 전, 답, 임야, 과수원, 목장용지
    if s in ("주/상용건물",):
        return "기타주거"
    return "상가·업무"                        # 근린생활, 업무, 숙박, 판매, 오피스텔, 기타용도복합 등
df["usage6"] = [usage6(m, s) for m, s in zip(df["cltrUsgMclsCtgrNm"], df["cltrUsgSclsCtgrNm"])]
df["prpt"] = df["prptDivNm"].map({"압류재산": "압류", "기타일반재산": "기타일반"}).fillna("기타")
df["opbd_qtr"] = df["opbd_date"].dt.quarter        # 계절만 (연도는 외삽 위험이라 안 씀)

print("\n유형 × 용도6 건수 / 낙찰가율 중앙값")
print(df.pivot_table(index="usage6", columns="prpt", values="y_ratio", aggfunc=["count", "median"]).round(3).to_string())

입력: train_features_20261105.csv (4772, 53)
중복·결측·극단값 정리: 4772 → 4736 → 4694건

유형 × 용도6 건수 / 낙찰가율 중앙값
         count        median       
prpt      기타일반     압류   기타일반     압류
usage6                             
공장·창고     57.0   26.0  0.684  0.363
기타주거     795.0  447.0  0.585  0.509
농지·임야      NaN  693.0    NaN  0.226
대지         NaN  216.0    NaN  0.312
상가·업무   1329.0  328.0  0.560  0.363
아파트      604.0  199.0  0.743  0.752


## 2단계. 지가변동률·지가지수 붙이기
매각 월의 **2개월 전** 값(발표 시차 때문에 누수 방지)과 그 전 6개월 누적

In [27]:
SIDO_SHORT = {"서울특별시": "서울", "부산광역시": "부산", "대구광역시": "대구", "인천광역시": "인천", "광주광역시": "광주",
              "대전광역시": "대전", "울산광역시": "울산", "세종특별자치시": "세종", "경기도": "경기",
              "강원특별자치도": "강원", "강원도": "강원", "충청북도": "충북", "충청남도": "충남",
              "전북특별자치도": "전북", "전라북도": "전북", "전라남도": "전남", "경상북도": "경북",
              "경상남도": "경남", "제주특별자치도": "제주"}

def read_rone(fname, kind):
    """R-ONE 넓은 표 → (시도, 시군구, 구, 월, 값) 긴 표"""
    t = pd.read_csv(os.path.join(LAND_DIR, fname), encoding="cp949", header=None)
    hdr_m, hdr_k = t.iloc[0], t.iloc[1]
    body = t.iloc[3:] if kind == "rate" else t.iloc[2:]
    rows = []
    for j in range(4, t.shape[1]):
        if kind == "rate" and str(hdr_k[j]) != "변동률":
            continue
        ym = pd.to_datetime(str(hdr_m[j]).replace("년 ", "-").replace("월", ""), format="%Y-%m", errors="coerce")
        if pd.isna(ym):
            continue
        part = pd.DataFrame({"a": body[1].values, "b": body[2].values, "c": body[3].values,
                             "ym": ym.to_period("M"), "v": pd.to_numeric(body[j], errors="coerce").values})
        rows.append(part)
    return pd.concat(rows, ignore_index=True)

def region_keys(sido, sgg):
    """학습 데이터의 (시도, 시군구) → R-ONE에서 찾을 후보 키 (구 → 시 → 시도 순)"""
    a = SIDO_SHORT.get(sido, sido)
    if not isinstance(sgg, str) or not sgg.strip():
        return [(a, a, a)]
    parts = sgg.split()
    if len(parts) >= 2:                      # 예: 안양시 동안구
        return [(a, parts[0], parts[1]), (a, parts[0], parts[0]), (a, a, a)]
    return [(a, parts[0], parts[0]), (a, a, a)]

def attach_market(frame, lag=2):
    if LAND_DIR is None:
        frame["lr_1m"] = np.nan; frame["lr_6m"] = np.nan
        return frame
    rate = read_rone("(월) 지역별 지가변동률.csv", "rate")
    idx = read_rone("(월) 지역별 지가지수.csv", "index")
    rate_d = {(r.a, r.b, r.c, r.ym): r.v for r in rate.itertuples()}
    idx_d = {(r.a, r.b, r.c, r.ym): r.v for r in idx.itertuples()}
    lr1, lr6, hit = [], [], []
    for sido, sgg, d in zip(frame["sido"], frame["sgg"], frame["opbd_date"]):
        m = d.to_period("M") - lag
        v1 = v6 = None; lvl = None
        for k, key in enumerate(region_keys(sido, sgg)):
            if v1 is None and (key + (m,)) in rate_d:
                v1 = rate_d[key + (m,)]; lvl = k
            if v6 is None and (key + (m,)) in idx_d and (key + (m - 6,)) in idx_d:
                v6 = (idx_d[key + (m,)] / idx_d[key + (m - 6,)] - 1) * 100
        lr1.append(v1); lr6.append(v6); hit.append(lvl)
    frame["lr_1m"] = pd.to_numeric(pd.Series(lr1, index=frame.index), errors="coerce")   # 2개월 전 월간 변동률(%)
    frame["lr_6m"] = pd.to_numeric(pd.Series(lr6, index=frame.index), errors="coerce")   # 그 전 6개월 누적(%)
    h = pd.Series(hit).map({0: "시군구/구", 1: "시", 2: "시도"}).fillna("없음")
    print("지가변동률 연결 수준:"); print(h.value_counts().to_string())
    return frame

df = attach_market(df)
print(df[["lr_1m", "lr_6m"]].describe().round(3).to_string())

지가변동률 연결 수준:
시군구/구    4207
없음        400
시          87
          lr_1m     lr_6m
count  4294.000  4278.000
mean      0.118     0.728
std       0.113     0.630
min      -0.107    -0.526
25%       0.047     0.336
50%       0.095     0.591
75%       0.180     1.113
max       0.626     3.270


## 3단계. 피처와 분할
train ~2026-08 / val 2026-09 / test 2026-10

In [28]:
CAT = ["usage6", "sido_grp", "bld_strct", "bld_purps"] + (["prpt"] if USE_PRPT else [])
NUM = ["log_apsl", "landSqms", "bldSqms", "has_building", "log_land_price", "apsl_to_official",
       "bld_age_c", "bld_grnd_flr", "bld_tot_area", "is_daegu_gb", "opbd_qtr", "lr_1m", "lr_6m"]

def split_of(d):
    if d <= pd.Timestamp(TRAIN_END): return "train"
    if d <= pd.Timestamp(VAL_END):   return "val"
    if d <= pd.Timestamp(TEST_END):  return "test"
    return "after"
df["split"] = df["opbd_date"].map(split_of)

def make_X(frame, cats):
    X = frame[NUM].apply(pd.to_numeric, errors="coerce").astype(float)
    for c in cats:
        X[c] = frame[c].astype(str).replace({"nan": "없음"}).astype("category")
    return X

tr, va, te = (df[df.split == s] for s in ["train", "val", "test"])
print(pd.crosstab(df["split"], df["prpt"]).to_string())
y_tr, y_va, y_te = tr["y_ratio"].values, va["y_ratio"].values, te["y_ratio"].values

prpt   기타일반   압류
split           
test     81  898
train  2635  708
val      69  303


## 4단계. 기준선 + 선형 분위수 회귀

In [29]:
def pinball(y, q_pred, q):
    e = y - q_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

def evaluate(name, frame, y, P):
    """P: (n, 3) = P10, P50, P90 (낙찰가율)"""
    P = np.sort(P, axis=1)                                   # 분위수 역전 방지
    out = {"모델": name, "n": len(y),
           "MAE(P50, %p)": np.mean(np.abs(y - P[:, 1])) * 100,
           "포함률(P10~P90)": np.mean((y >= P[:, 0]) & (y <= P[:, 2])),
           "구간폭(%p)": np.mean(P[:, 2] - P[:, 0]) * 100,
           "pinball": np.mean([pinball(y, P[:, i], q) for i, q in enumerate(QS)])}
    return out

def by_group(frame, y, P, col):
    P = np.sort(P, axis=1)
    g = pd.DataFrame({"g": frame[col].values, "y": y, "p10": P[:, 0], "p50": P[:, 1], "p90": P[:, 2]})
    return g.groupby("g").apply(lambda s: pd.Series({
        "n": len(s), "MAE(%p)": np.mean(np.abs(s.y - s.p50)) * 100,
        "포함률": np.mean((s.y >= s.p10) & (s.y <= s.p90))})).round(3)

# 기준선: 같은 (유형×)용도의 train 낙찰가율 10/50/90% 분위수를 그대로 예측 (ML 아님)
def baseline_predict(train, target, keys):
    tab = train.groupby(keys)["y_ratio"].quantile(QS).unstack()
    glob_q = train["y_ratio"].quantile(QS).values
    P = []
    for _, r in target[keys].iterrows():
        k = tuple(r) if len(keys) > 1 else r.iloc[0]
        P.append(tab.loc[k].values if k in tab.index else glob_q)
    return np.array(P)

base_keys = ["prpt", "usage6"] if USE_PRPT else ["usage6"]
P_base_va = baseline_predict(tr, va, base_keys)

# 선형 분위수 회귀 (statsmodels QuantReg) — 숫자는 결측을 train 중앙값으로, 범주는 더미
import statsmodels.api as sm
def linear_design(frame, ref=None):
    num = ["log_apsl", "log_land_price", "has_building", "bld_age_c", "lr_1m"]
    X = frame[num].apply(pd.to_numeric, errors="coerce")
    med = X.median() if ref is None else ref["med"]
    X = X.fillna(med)
    for c in num:                                             # 결측 여부 표시
        X[c + "_na"] = frame[c].isna().astype(float)
    cats = ["usage6"] + (["prpt"] if USE_PRPT else [])
    D = pd.get_dummies(frame[cats].astype(str), drop_first=False, dtype=float)
    X = pd.concat([X, D], axis=1)
    cols = X.columns if ref is None else ref["cols"]
    X = X.reindex(columns=cols, fill_value=0.0)
    X = X.loc[:, X.std() > 0] if ref is None else X
    return X, {"med": med, "cols": X.columns}

Xl_tr, ref = linear_design(tr)
Xl_va, _ = linear_design(va, ref)
P_lin_va = np.column_stack([sm.QuantReg(y_tr, sm.add_constant(Xl_tr, has_constant="add"))
                            .fit(q=q, max_iter=5000).predict(sm.add_constant(Xl_va, has_constant="add"))
                            for q in QS])

res = [evaluate("기준선", va, y_va, P_base_va), evaluate("선형 분위수 회귀", va, y_va, P_lin_va)]
print(pd.DataFrame(res).round(3).to_string(index=False))

       모델   n  MAE(P50, %p)  포함률(P10~P90)  구간폭(%p)  pinball
      기준선 372        22.080         0.731   62.284    0.068
선형 분위수 회귀 372        22.172         0.774   65.865    0.067


## 5단계. LightGBM 분위수 회귀 (P10/P50/P90)
격자 3개 × 재산유형 있음/없음을 val로 비교해요. 몇 분 걸려요.

In [30]:
import lightgbm as lgb

GRID = [dict(num_leaves=15, min_child_samples=40, learning_rate=0.03),
        dict(num_leaves=31, min_child_samples=20, learning_rate=0.03),
        dict(num_leaves=7,  min_child_samples=60, learning_rate=0.05)]

def fit_lgb(train, valid, cats, params):
    Xtr, Xva = make_X(train, cats), make_X(valid, cats)
    for c in cats:                                           # val 범주를 train 범주에 맞춤
        Xva[c] = pd.Categorical(Xva[c].astype(str), categories=Xtr[c].cat.categories)
    models, P = {}, []
    for q in QS:
        m = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=3000, subsample=0.8,
                              subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                              random_state=SEED, verbose=-1, **params)
        m.fit(Xtr, train["y_ratio"], eval_set=[(Xva, valid["y_ratio"])], eval_metric="quantile",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        models[q] = m
        P.append(m.predict(Xva))
    return models, np.column_stack(P), Xtr.columns

best = None
for prpt_flag in ([True, False]):                            # 재산유형 피처 있을 때 / 없을 때 비교
    cats = ["usage6", "sido_grp", "bld_strct", "bld_purps"] + (["prpt"] if prpt_flag else [])
    for i, params in enumerate(GRID):
        models, P, cols = fit_lgb(tr, va, cats, params)
        r = evaluate(f"LightGBM 격자{i} 유형{'O' if prpt_flag else 'X'}", va, y_va, P)
        res.append(r)
        if prpt_flag == USE_PRPT and (best is None or r["pinball"] < best["r"]["pinball"]):
            best = {"r": r, "params": params, "models": models, "P": P, "cats": cats}

print(pd.DataFrame(res).round(3).to_string(index=False))
print("\n선택:", best["r"]["모델"], best["params"],
      "| 트리 수", {q: m.best_iteration_ for q, m in best["models"].items()})

              모델   n  MAE(P50, %p)  포함률(P10~P90)  구간폭(%p)  pinball
             기준선 372        22.080         0.731   62.284    0.068
       선형 분위수 회귀 372        22.172         0.774   65.865    0.067
LightGBM 격자0 유형O 372        20.215         0.645   55.047    0.063
LightGBM 격자1 유형O 372        19.540         0.610   52.674    0.064
LightGBM 격자2 유형O 372        20.714         0.672   57.683    0.064
LightGBM 격자0 유형X 372        20.882         0.653   55.053    0.067
LightGBM 격자1 유형X 372        19.983         0.634   52.828    0.066
LightGBM 격자2 유형X 372        21.412         0.688   56.086    0.068

선택: LightGBM 격자0 유형O {'num_leaves': 15, 'min_child_samples': 40, 'learning_rate': 0.03} | 트리 수 {0.1: 77, 0.5: 194, 0.9: 468}


## 6단계. 구간 보정(CQR) + val 상세
★ 신보 용도(공장·창고, 상가·업무, 대지, 농지·임야) 성적표가 대표 숫자예요.

In [31]:
# val에서 실제값이 구간 밖으로 벗어난 정도의 80% 분위수만큼 구간을 넓힘(음수면 좁힘)
def cqr_delta(y, P):
    s_ = np.maximum(P[:, 0] - y, y - P[:, 2]); n_ = len(y)
    return float(np.quantile(s_, min(1.0, np.ceil((n_ + 1) * 0.8) / n_)))

def apply_cqr(frame, P):
    P = np.sort(P, axis=1).copy()
    d = frame["usage6"].map(CQR_USAGE).fillna(CQR_DELTA).values if CQR_BY_USAGE else CQR_DELTA
    P[:, 0] -= d; P[:, 2] += d
    return P

P_va = np.sort(best["P"], axis=1)
CQR_DELTA = cqr_delta(y_va, P_va)
CQR_USAGE = {}
for u, g in va.groupby("usage6"):
    if len(g) >= 30:
        m = (va["usage6"] == u).values
        CQR_USAGE[u] = cqr_delta(y_va[m], P_va[m])
print(f"CQR 전체 보정폭: {CQR_DELTA*100:+.2f}%p")
print("용도별 보정폭(%p):", {k: round(v * 100, 2) for k, v in CQR_USAGE.items()}, "| 나머지 용도는 전체 보정폭")
print("※ val로 구했으므로 val 포함률은 정의상 약 80%. 진짜 확인은 test")
P_va_cqr = apply_cqr(va, P_va)

print("\n[val] 보정 전/후")
print(pd.DataFrame([evaluate("LightGBM", va, y_va, P_va),
                    evaluate("LightGBM + CQR", va, y_va, P_va_cqr)]).round(3).to_string(index=False))
m_sb = va["usage6"].isin(SINBO_USAGE).values
print("\n[val] ★ 신보 용도만 (공장·창고, 상가·업무, 대지, 농지·임야)")
print(pd.DataFrame([evaluate("기준선", va[m_sb], y_va[m_sb], P_base_va[m_sb]),
                    evaluate("LightGBM + CQR", va[m_sb], y_va[m_sb], P_va_cqr[m_sb])]).round(3).to_string(index=False))
print("\n[val] 재산유형별"); print(by_group(va, y_va, P_va_cqr, "prpt").to_string())
print("\n[val] 용도별");     print(by_group(va, y_va, P_va_cqr, "usage6").to_string())

imp = pd.Series(best["models"][0.5].feature_importances_, index=make_X(tr, best["cats"]).columns)
print("\n[P50 모델] 피처 중요도 (분할 횟수)"); print(imp.sort_values(ascending=False).to_string())

CQR 전체 보정폭: +6.10%p
용도별 보정폭(%p): {'기타주거': 9.79, '농지·임야': 3.75, '대지': 3.56, '상가·업무': 8.6, '아파트': -0.65} | 나머지 용도는 전체 보정폭
※ val로 구했으므로 val 포함률은 정의상 약 80%. 진짜 확인은 test

[val] 보정 전/후
            모델   n  MAE(P50, %p)  포함률(P10~P90)  구간폭(%p)  pinball
      LightGBM 372        20.215         0.645   55.047    0.063
LightGBM + CQR 372        20.215         0.817   66.211    0.061

[val] ★ 신보 용도만 (공장·창고, 상가·업무, 대지, 농지·임야)
            모델   n  MAE(P50, %p)  포함률(P10~P90)  구간폭(%p)  pinball
           기준선 256        21.922         0.727   65.563    0.069
LightGBM + CQR 256        21.497         0.816   68.747    0.065

[val] 재산유형별
          n  MAE(%p)    포함률
g                          
기타일반   69.0   14.294  0.870
압류    303.0   21.563  0.805

[val] 용도별
           n  MAE(%p)    포함률
g                           
공장·창고    4.0   13.173  1.000
기타주거    75.0   19.319  0.813
농지·임야  122.0   21.911  0.811
대지      46.0   24.684  0.826
상가·업무   84.0   19.546  0.810
아파트     41.0   13.848  0.829

[P50 모델] 피처 중요도 (분할 

## 7단계. test 평가 — ⚠️ 마지막에 딱 한 번
val로 설정을 다 정한 뒤 `RUN_TEST = True`로 바꿔 실행하세요. 결과를 보고 모델을 고치면 test가 의미 없어져요.

In [34]:
RUN_TEST = False

if RUN_TEST:
    Xte = make_X(te, best["cats"]); Xtr_ = make_X(tr, best["cats"])
    for c in best["cats"]:
        Xte[c] = pd.Categorical(Xte[c].astype(str), categories=Xtr_[c].cat.categories)
    P_te = apply_cqr(te, np.column_stack([best["models"][q].predict(Xte) for q in QS]))
    P_base_te = baseline_predict(tr, te, base_keys)
    Xl_te, _ = linear_design(te, ref)
    P_lin_te = np.column_stack([sm.QuantReg(y_tr, sm.add_constant(Xl_tr, has_constant="add")).fit(q=q, max_iter=5000)
                                .predict(sm.add_constant(Xl_te, has_constant="add")) for q in QS])
    print(pd.DataFrame([evaluate("기준선", te, y_te, P_base_te), evaluate("선형", te, y_te, P_lin_te),
                        evaluate("LightGBM + CQR", te, y_te, P_te)]).round(3).to_string(index=False))
    m_sb = te["usage6"].isin(SINBO_USAGE).values
    print("\n[test] ★ 신보 용도만")
    print(pd.DataFrame([evaluate("기준선", te[m_sb], y_te[m_sb], P_base_te[m_sb]),
                        evaluate("LightGBM + CQR", te[m_sb], y_te[m_sb], P_te[m_sb])]).round(3).to_string(index=False))
    print("\n[test] 재산유형별"); print(by_group(te, y_te, P_te, "prpt").to_string())
    print("\n[test] 용도별");     print(by_group(te, y_te, P_te, "usage6").to_string())
    print("\n[test] 대구·경북");   print(by_group(te, y_te, P_te, "is_daegu_gb").to_string())
    out = te[["cltrMngNo", "onbidCltrNm", "prpt", "usage6", "sido", "opbd_date", "apslEvlAmt", "y_ratio"]].copy()
    out[["p10", "p50", "p90"]] = P_te
    out.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False, encoding="utf-8-sig")
    print("저장: model_out/test_predictions.csv")
else:
    print("test는 아직 실행하지 않음 (val로 설정을 다 정한 뒤 RUN_TEST = True)")

            모델   n  MAE(P50, %p)  포함률(P10~P90)  구간폭(%p)  pinball
           기준선 979        22.215         0.757   63.364    0.067
            선형 979        21.549         0.799   66.826    0.066
LightGBM + CQR 979        17.317         0.855   66.169    0.055

[test] ★ 신보 용도만
            모델   n  MAE(P50, %p)  포함률(P10~P90)  구간폭(%p)  pinball
           기준선 622        22.122         0.736   65.190    0.067
LightGBM + CQR 622        18.809         0.860   67.993    0.058

[test] 재산유형별
          n  MAE(%p)    포함률
g                          
기타일반   81.0   13.821  0.778
압류    898.0   17.633  0.862

[test] 용도별
           n  MAE(%p)    포함률
g                           
공장·창고   18.0   21.354  0.889
기타주거   234.0   15.763  0.876
농지·임야  312.0   19.291  0.856
대지      98.0   20.889  0.908
상가·업무  194.0   16.748  0.840
아파트    123.0   12.730  0.789

[test] 대구·경북
           n  MAE(%p)    포함률
g                           
False  901.0   17.031  0.855
True    78.0   20.629  0.859
저장: model_out/test_predictio

## 8단계. 최종 모델 저장
test까지 끝난 뒤 `FINAL = True`. 결과는 `data/model_out/`에 저장돼요.

In [35]:
FINAL = False

if FINAL:
    full = df[df.split != "after"]
    Xf = make_X(full, best["cats"])
    final = {}
    for q in QS:
        n_iter = max(50, int(best["models"][q].best_iteration_ * len(full) / len(tr)))   # 데이터가 늘어난 만큼 트리 수 비례
        m = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=n_iter, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, verbose=-1, **best["params"])
        final[q] = m.fit(Xf, full["y_ratio"])
        m.booster_.save_model(os.path.join(OUT_DIR, f"lgb_q{int(q*100)}.txt"))
    meta = {"features_num": NUM, "features_cat": best["cats"],
            "categories": {c: list(Xf[c].cat.categories) for c in best["cats"]},
            "cqr_delta": CQR_DELTA, "cqr_by_usage": CQR_USAGE if CQR_BY_USAGE else {}, "params": best["params"], "use_prpt": USE_PRPT,
            "note": "경매 물건 예측 시 prpt='압류'로 고정, 이후 공매→경매 보정 계수 곱함"}
    json.dump(meta, open(os.path.join(OUT_DIR, "model_meta.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("저장: model_out/lgb_q10/q50/q90.txt, model_meta.json")
else:
    print("최종 모델은 test까지 끝난 뒤 FINAL = True")

저장: model_out/lgb_q10/q50/q90.txt, model_meta.json
